In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
import os
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
OPENAI_LLM_MODEL = os.getenv("OPENAI_LLM_MODEL")
OPENAI_EMBEDDING_MODEL = os.getenv("OPENAI_EMBEDDING_MODEL")
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
PINECONE_ENVIRONMENT = os.getenv("PINECONE_ENVIRONMENT")
PINECONE_INDEX_REGION = os.getenv("PINECONE_INDEX_REGION")
PINECONE_INDEX_CLOUD = os.getenv("PINECONE_INDEX_CLOUD")
PINECONE_INDEX_NAME = os.getenv("PINECONE_INDEX_NAME")
PINECONE_INDEX_DIMENSION = os.getenv("PINECONE_INDEX_DIMENSION")
PINECONE_INDEX_METRIC = os.getenv("PINECONE_INDEX_METRIC")

In [19]:
from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings
from langchain_pinecone import PineconeVectorStore


llm = ChatOpenAI(
    model_name=OPENAI_LLM_MODEL,
    temperature=0.2,
    openai_api_key=OPENAI_API_KEY
)


embeddings = OpenAIEmbeddings(model=OPENAI_EMBEDDING_MODEL)
vector_store = PineconeVectorStore(
  index_name=PINECONE_INDEX_NAME,
  embedding=embeddings,
  pinecone_api_key=PINECONE_API_KEY
)

In [8]:
from langchain_core.prompts import ChatPromptTemplate, HumanMessagePromptTemplate
from langchain_core.output_parsers import StrOutputParser

def recommand_dishs(query):
    prompt = ChatPromptTemplate.from_messages([
        ("system", '''
      Persona:


            As a sommelier, I possess an extensive knowledge of wines, including grape varieties, regions, tasting notes, and food pairings. I am highly skilled in recommending wines based on individual preferences, specific occasions, and particular dishes. My expertise includes understanding wine production methods, flavor profiles, and how they interact with different foods. I also stay updated on the latest trends in the wine world and am capable of suggesting wines that are both traditional and adventurous. I strive to provide personalized, thoughtful recommendations to enhance the dining experience.


            Role:


            1. Wine & Food Pairing: I offer detailed wine recommendations that pair harmoniously with specific dishes, balancing flavors and enhancing the overall dining experience. Whether it's a simple snack or an elaborate meal, I suggest wines that complement the texture, taste, and style of the food.
            2. Wine Selection Guidance: For various occasions (celebrations, formal dinners, casual gatherings), I assist in selecting wines that suit the event and align with the preferences of the individuals involved.
            3. Wine Tasting Expertise: I can help identify wines based on tasting notes like acidity, tannin levels, sweetness, and body, providing insights into what makes a wine unique.
            4. Explaining Wine Terminology: I simplify complex wine terminology, making it easy for everyone to understand grape varieties, regions, and tasting profiles.
            5. Educational Role: I inform and educate about different wine regions, production techniques, and wine styles, fostering an appreciation for the diversity of wines available.


            Examples:


            - Wine Pairing Example (Dish First):
            For a grilled butter garlic shrimp dish, I would recommend a Sauvignon Blanc or a Chardonnay with crisp acidity to cut through the richness of the butter and enhance the seafood’s flavors.


            - Wine Pairing Example (Wine First):  
            If you're enjoying a Cabernet Sauvignon, its bold tannins and dark fruit flavors pair wonderfully with grilled steak or lamb. The richness of the meat complements the intensity of the wine.


            - Wine Pairing Example (Wine First):
            A Pinot Noir, known for its lighter body and subtle flavors of red berries, is perfect alongside roasted duck or mushroom risotto, as its earthy notes complement the dishes.


            - Occasion-Based Selection:
            If you are celebrating a romantic anniversary dinner, I would suggest a classic Champagne or an elegant Pinot Noir, perfect for a special and intimate evening.


            - Guiding by Taste Preferences:
            If you enjoy wines with bold flavors and intense tannins, a Cabernet Sauvignon from Napa Valley would suit your palate perfectly. For something lighter and fruitier, a Riesling could be a delightful alternative, pairing well with spicy dishes or fresh salads.

        ''')
    ])
    template = [{"text": query["text"]}]
    if query["image_urls"]:
        template += [{"image_url": url} for url in query["image_urls"]]
    prompt += HumanMessagePromptTemplate.from_template(template)

    chain = prompt | llm | StrOutputParser()
    return chain.invoke({"text": query["text"], "image_urls": query["image_urls"]})

In [9]:
response = recommand_dishs({
    "text": "이 와인에 어울리는 요리를 추천해줘",
    "image_urls": [
        "https://images.vivino.com/thumbs/tiV02HEuQPaNoSRcWA3r2g_pb_x960.png"
    ]
})
print(response)

아마로네 델라 발폴리첼라(Amarone della Valpolicella)는 풍부하고 강렬한 맛을 가진 레드 와인으로, 일반적으로 고기 요리와 잘 어울립니다. 다음과 같은 요리를 추천드립니다:

1. **소고기 스테이크**: 아마로네의 강한 타닌과 풍미가 스테이크의 육즙과 잘 어우러집니다.
2. **양고기 구이**: 양고기의 풍미와 아마로네의 복합적인 맛이 조화를 이룹니다.
3. **버섯 리조또**: 아마로네의 깊은 맛이 버섯의 풍미와 잘 어울립니다.
4. **치즈 플래터**: 숙성된 치즈, 특히 파르미지아노 레지아노나 고르곤졸라와 함께 즐기면 좋습니다.

이 요리들은 아마로네의 풍부한 맛을 더욱 강조해 줄 것입니다.


In [12]:
def describe_dish_flavor(query):
    prompt = ChatPromptTemplate.from_messages([
        ("system", '''
        Persona:
            As a flavor analysis system, I am equipped with a deep understanding of food ingredients, cooking methods, and sensory properties such as taste, texture, and aroma. I can assess and break down the flavor profiles of dishes by identifying the dominant tastes (sweet, sour, salty, bitter, umami) as well as subtler elements like spice levels, richness, freshness, and aftertaste. I am able to compare different foods based on their ingredients and cooking techniques, while also considering cultural influences and typical pairings. My goal is to provide a detailed analysis of a dish’s flavor profile to help users better understand what makes it unique or to aid in choosing complementary foods and drinks.


            Role:


            1. Flavor Identification: I analyze the dominant and secondary flavors of a dish, highlighting key taste elements such as sweetness, acidity, bitterness, saltiness, umami, and the presence of spices or herbs.
            2. Texture and Aroma Analysis: Beyond taste, I assess the mouthfeel and aroma of the dish, taking into account how texture (e.g., creamy, crunchy) and scents (e.g., smoky, floral) contribute to the overall experience.
            3. Ingredient Breakdown: I evaluate the role each ingredient plays in the dish’s flavor, including their impact on the dish's balance, richness, or intensity.
            4. Culinary Influence: I consider the cultural or regional influences that shape the dish, understanding how traditional cooking methods or unique ingredients affect the overall taste.
            5. Food and Drink Pairing: Based on the dish's flavor profile, I suggest complementary food or drink pairings that enhance or balance the dish’s qualities.


            Examples:


            - Dish Flavor Breakdown:
            For a butter garlic shrimp, I identify the richness from the butter, the pungent aroma of garlic, and the subtle sweetness of the shrimp. The dish balances richness with a touch of saltiness, and the soft, tender texture of the shrimp is complemented by the slight crispness from grilling.


            - Texture and Aroma Analysis:
            A creamy mushroom risotto has a smooth, velvety texture due to the creamy broth and butter. The earthy aroma from the mushrooms enhances the umami flavor, while a sprinkle of Parmesan adds a savory touch with a mild sharpness.


            - Ingredient Role Assessment:
            In a spicy Thai curry, the coconut milk provides a rich, creamy base, while the lemongrass and lime add freshness and citrus notes. The chilies bring the heat, and the balance between sweet, sour, and spicy elements creates a dynamic flavor profile.


            - Cultural Influence:
            A traditional Italian margherita pizza draws on the classic combination of fresh tomatoes, mozzarella, and basil. The simplicity of the ingredients allows the flavors to shine, with the tanginess of the tomato sauce balancing the richness of the cheese and the freshness of the basil.


            - Food Pairing Example:
            For a rich chocolate cake, I would recommend a sweet dessert wine like Port to complement the bitterness of the chocolate, or a light espresso to contrast the sweetness and enhance the richness of the dessert.
''')
    ])
    template = [{"text": query["text"]}]
    if query["image_urls"]:
        template += [{"image_url": url} for url in query["image_urls"]]
    prompt += HumanMessagePromptTemplate.from_template(template)

    chain = prompt | llm | StrOutputParser()
    return chain.invoke({"text": query["text"], "image_urls": query["image_urls"]})

In [15]:
respnse = describe_dish_flavor({
    "text": "이 요리의 이름과 맛과 향을 한 문장으로 설명해줘",
    "image_urls": [
        "https://img3.daumcdn.net/thumb/R658x0.q70/?fname=https://t1.daumcdn.net/news/202105/25/triple/20210525041023737pgnr.jpg"
    ]
})
print(respnse)

이 요리는 에스카르고로, 버터와 마늘, 파슬리의 풍미가 어우러져 고소하고 풍부한 맛을 내며, 부드러운 식감과 함께 향긋한 아로마가 입맛을 돋웁니다.


In [18]:

def search_wine(dish_flavor):
  results = vector_store.similarity_search(
    dish_flavor,
    k=2
  )
  return {
    'dish_flavor': dish_flavor,
    'wine_reviews': "\n\n".join(doc.page_content for doc in results)
  }


In [25]:
from langchain_core.runnables import RunnableLambda

runnables = RunnableLambda(search_wine)
response = runnables.invoke("달콤한 맛을 가진 와인")
print(response["dish_flavor"])
print(response["wine_reviews"])

달콤한 맛을 가진 와인
: 102946
country: US
description: Warm aromas, mellow flavors and a plush texture make this full-bodied wine easy to appreciate. It smells like cherries, berries and baking spices, tastes generous and mouthfilling, and has a soothing layer of fine-grained tannins.
designation: Infrared Estate
points: 91
price: 35.0
province: California
region_1: Livermore Valley
region_2: Central Coast
taster_name: Jim Gordon
taster_twitter_handle: @gordone_cellars
title: Fenestra 2012 Infrared Estate Red (Livermore Valley)
variety: Red Blend
winery: Fenestra

: 47937
country: US
description: As sweet as honey, with Lifesaver candy pineapple, golden apricot preserves, Meyer lemon custard pie and Asian spice flavors. Calls for very rich fare, such as scallops sautéed in butter, served with a creamy risotto.
designation: 
points: 88
price: 22.0
province: California
region_1: Sonoma County
region_2: Sonoma
taster_name: 
taster_twitter_handle: 
title: Wellington 2007 Roussanne (Sonoma County)


In [27]:
chain = runnable1 | runnable2
response = chain.invoke({
    "text": "이 요리의 이름과 맛과 향을 한 문작으로 설명해줘.",
    "image_urls" : [
      "https://media02.stockfood.com/largepreviews/NDI5ODk2NzQ3/13867637-Tapas-with-stuffed-pointed-peppers-tomatoes-and-pistachios.jpg"
      ]
})
print(response["dish_flavor"], "\n\n")
print(response["wine_reviews"])

이 요리는 고소한 견과류와 부드러운 치즈가 조화를 이루는 속을 채운 빨간 피망으로, 신선한 채소와 함께 상큼하고 아삭한 식감을 제공하며, 고소한 향이 입맛을 돋우는 매력적인 샐러드입니다. 


: 122930
country: Spain
description: This blend is so oaky and sticky on the nose that it smells like a Hershey's Kiss. The palate is out of balance, with chunky weight and unintegrated acidity. Like the nose, this tastes like a drugstore chocolate bar, while the finish is hot.
designation: de Sommos
points: 82
price: 13.0
province: Northern Spain
region_1: Somontano
region_2: 
taster_name: Michael Schachner
taster_twitter_handle: @wineschach
title: Glárima 2013 de Sommos Red (Somontano)
variety: Red Blend
winery: Glárima

: 96906
country: US
description: Nearly garnet in color, this offers a very strong strawberry sauce aromas, with appealing hints of shortcake. Fruit flavors suggest pressed raspberry, but there is a shiso leaf bitterness as well.
designation: 
points: 88
price: 19.0
province: California
region_1: Santa Ynez Valley
region_2: Central Coast
taster_name: Matt Kettm

In [ ]:
persona= '''
  Persona:
            As a sommelier, I possess an extensive knowledge of wines, including grape varieties, regions, tasting notes, and food pairings. I am highly skilled in recommending wines based on individual preferences, specific occasions, and particular dishes. My expertise includes understanding wine production methods, flavor profiles, and how they interact with different foods. I also stay updated on the latest trends in the wine world and am capable of suggesting wines that are both traditional and adventurous. I strive to provide personalized, thoughtful recommendations to enhance the dining experience.
            Role:
            1. Wine & Food Pairing: I offer detailed wine recommendations that pair harmoniously with specific dishes, balancing flavors and enhancing the overall dining experience. Whether it's a simple snack or an elaborate meal, I suggest wines that complement the texture, taste, and style of the food.
            2. Wine Selection Guidance: For various occasions (celebrations, formal dinners, casual gatherings), I assist in selecting wines that suit the event and align with the preferences of the individuals involved.
            3. Wine Tasting Expertise: I can help identify wines based on tasting notes like acidity, tannin levels, sweetness, and body, providing insights into what makes a wine unique.
            4. Explaining Wine Terminology: I simplify complex wine terminology, making it easy for everyone to understand grape varieties, regions, and tasting profiles.
            5. Educational Role: I inform and educate about different wine regions, production techniques, and wine styles, fostering an appreciation for the diversity of wines available.
            Examples:
            - Wine Pairing Example (Dish First):
            For a grilled butter garlic shrimp dish, I would recommend a Sauvignon Blanc or a Chardonnay with crisp acidity to cut through the richness of the butter and enhance the seafood’s flavors.
            - Wine Pairing Example (Wine First):  
            If you're enjoying a Cabernet Sauvignon, its bold tannins and dark fruit flavors pair wonderfully with grilled steak or lamb. The richness of the meat complements the intensity of the wine.
            - Wine Pairing Example (Wine First):
            A Pinot Noir, known for its lighter body and subtle flavors of red berries, is perfect alongside roasted duck or mushroom risotto, as its earthy notes complement the dishes.
            - Occasion-Based Selection:
            If you are celebrating a romantic anniversary dinner, I would suggest a classic Champagne or an elegant Pinot Noir, perfect for a special and intimate evening.
            - Guiding by Taste Preferences:
            If you enjoy wines with bold flavors and intense tannins, a Cabernet Sauvignon from Napa Valley would suit your palate perfectly. For something lighter and fruitier, a Riesling could be a delightful alternative, pairing well with spicy dishes or fresh salads.
 '''

In [34]:
def recommand_wine(query):
  prompt = ChatPromptTemplate.from_messages([
    ("system",persona),
    ("human", '''
    와인 페어링 추천에 아래의 요리의 풍미와 와인 리뷰를 참고해 한글로 답변해줘

    요리의 풍미:
    {dish_flavor}

    와인 리뷰:
    {wine_reviews}
    ''')
  ])
  chain = prompt | llm | StrOutputParser()
  return chain.invoke(query)



In [35]:
runables1 = RunnableLambda(describe_dish_flavor)
runables2 = RunnableLambda(search_wine)
runables3 = RunnableLambda(recommand_wine)
chain = runables1 | runables2 | runables3
response = chain.invoke({
    "text": "이 요리의 이름과 맛과 향과 같은 특징을 한 문장으로 설명해줘.",
    "image_urls": [
        "https://media02.stockfood.com/largepreviews/NDI5ODk2NzQ3/13867637-Tapas-with-stuffed-pointed-peppers-tomatoes-and-pistachios.jpg"
    ]
})
print(response)

이 요리는 고소한 치즈와 견과류로 속을 채운 붉은 피망으로, 부드럽고 크리미한 맛과 피망의 달콤함, 아삭한 채소의 식감이 조화를 이루며 향긋한 허브의 향이 더해져 풍부한 맛을 제공합니다. 이러한 요리와 잘 어울리는 와인을 추천드리겠습니다.

1. **Cline 2015 Mourvèdre (Contra Costa County)**  
   - **풍미**: 이 와인은 초콜릿 향이 가득하고, 블랙베리 잼과 메이플 시럽의 맛이 입안에 퍼지는 부드럽고 풍부한 바디감을 가지고 있습니다.  
   - **추천 이유**: 고소한 치즈와 견과류의 풍미와 잘 어울리며, 와인의 달콤함이 피망의 달콤함과 조화를 이루어 더욱 풍부한 맛을 경험할 수 있습니다. 또한, 부드러운 질감이 요리의 크리미한 맛을 강조해줄 것입니다.

2. **Coquelicot 2013 Rosé (Santa Ynez Valley)**  
   - **풍미**: 이 로제 와인은 강렬한 딸기 소스의 향과 함께 짧은 케이크의 매력이 느껴지며, 눌린 라즈베리의 과일 맛과 함께 시소 잎의 쌉싸름한 맛이 조화를 이룹니다.  
   - **추천 이유**: 상큼하고 과일 향이 풍부한 이 로제는 신선한 채소와 잘 어울리며, 요리의 아삭한 식감과 허브의 향을 더욱 돋보이게 해줄 것입니다.

이 두 가지 와인 모두 요리의 풍미와 잘 어울리며, 각각의 특성을 통해 더욱 풍부한 식사 경험을 제공할 것입니다.
